# Gaia DR3: a million-star Hertzsprung–Russell diagram

This notebook queries the official ESA Gaia Archive and gives the raw
color–magnitude points to XY. The main sequence, red-giant branch, and
white-dwarf sequence emerge as density structure; no pre-binning is
required.

Gaia DR3 contains roughly 1.8 billion sources. The default query keeps
this run practical at one million high-quality stars. Increase
`GAIA_ROWS` to exercise a larger slice.

**Source:** [Gaia Archive programmatic access](https://www.cosmos.esa.int/web/gaia-users/archive/programmatic-access)
and [`gaiadr3.gaia_source`](https://gea.esac.esa.int/archive/documentation/GDR3/Gaia_archive/chap_datamodel/sec_dm_main_tables/ssec_dm_gaia_source.html).

Install beside XY with `python -m pip install numpy requests xy`.


In [ ]:
import os
from pathlib import Path

import numpy as np
import requests

import xy

DATA_DIR = Path(os.getenv("XY_REAL_WORLD_DATA", "data"))
DATA_DIR.mkdir(parents=True, exist_ok=True)

ROW_LIMIT = int(os.getenv("GAIA_ROWS", "1000000"))
if ROW_LIMIT <= 0:
    raise ValueError("GAIA_ROWS must be positive")

TAP_SYNC = "https://gea.esac.esa.int/tap-server/tap/sync"
query = f"""
SELECT TOP {ROW_LIMIT}
    bp_rp,
    phot_g_mean_mag,
    parallax
FROM gaiadr3.gaia_source
WHERE bp_rp IS NOT NULL
    AND phot_g_mean_mag IS NOT NULL
    AND parallax > 0
    AND parallax_over_error > 10
    AND phot_g_mean_flux_over_error > 50
"""

csv_path = DATA_DIR / f"gaia-dr3-hr-{ROW_LIMIT}.csv"
if not csv_path.exists():
    with requests.post(
        TAP_SYNC,
        data={
            "REQUEST": "doQuery",
            "LANG": "ADQL",
            "FORMAT": "csv",
            "QUERY": query,
        },
        stream=True,
        timeout=(30, 3600),
    ) as response:
        response.raise_for_status()
        partial = csv_path.with_suffix(".csv.part")
        with partial.open("wb") as output:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                output.write(chunk)
        partial.replace(csv_path)

print(f"cached query result: {csv_path}")

In [ ]:
stars = np.genfromtxt(
    csv_path,
    delimiter=",",
    names=True,
    dtype=np.float64,
    encoding="utf-8",
)
stars = np.atleast_1d(stars)

color_index = stars["bp_rp"]
absolute_g = stars["phot_g_mean_mag"] + 5 * np.log10(stars["parallax"]) - 10
finite = np.isfinite(color_index) & np.isfinite(absolute_g)
color_index = color_index[finite]
absolute_g = absolute_g[finite]

print(f"{color_index.size:,} stars")

In [ ]:
chart = xy.scatter_chart(
    xy.scatter(
        color_index,
        absolute_g,
        color=absolute_g,
        colormap="magma_r",
        size=1.0,
        opacity=0.7,
        density=True,
    ),
    xy.x_axis(label="Gaia BP - RP color (mag)", domain=(-1.0, 5.0)),
    xy.y_axis(
        label="Absolute G magnitude",
        domain=(-6.0, 16.0),
        reverse=True,
    ),
    xy.colorbar(title="absolute G"),
    xy.theme(
        background="#060812",
        text_color="#f8fafc",
        grid_color="#20263a",
        axis_color="#94a3b8",
    ),
    title=f"Gaia DR3 Hertzsprung-Russell diagram · {color_index.size:,} stars",
    width=1050,
    height=700,
)
print(chart.memory_report()["canonical_bytes"], "canonical bytes")
chart